In [ ]:
from pyGmsh import pyGmsh
import numpy as np
import openseespy.opensees as ops
import gmsh

# Shell I-Beam — Point Load + Linear Buckling (IPE200)

**Scenario:** Simply-supported IPE200 (3 m span), vertical point load
at midspan applied at the top flange. We perform:

1. **Static analysis** — deflection and deformed shape under a reference load.
2. **Linear buckling via VCT** — Vibration Correlation Technique: track how
   natural frequencies drop as load increases, then extrapolate to find the
   critical buckling load P_cr where ω² → 0.

**Why a shell model?**

Beam theory predicts global Euler buckling and gives closed-form LTB
(lateral-torsional buckling) estimates, but cannot capture **local** buckling
(flange or web buckling). A shell model captures all three simultaneously:
global flexural, lateral-torsional, and local buckling — no ad-hoc effective
length factors needed.

**VCT (Vibration Correlation Technique):**

Under increasing compressive/bending stress, geometric stiffness reduces
the effective stiffness, lowering natural frequencies. In the linear regime:

$$\omega^2(P) \approx \omega_0^2 \left(1 - \frac{P}{P_{cr}}\right)$$

Plotting ω² vs P gives a straight line whose x-intercept is P_cr.

In [ ]:
# ============================================================
# IPE200 cross-section
# ============================================================
#          ←── b ──→
#    ┌─────────────────┐  ─┬─ tf
#    └───────┐ ┌───────┘   │
#            │ │  tw        │ h = 200 mm
#            │ │            │
#    ┌───────┘ └───────┐   │
#    └─────────────────┘  ─┴─ tf
#
h   = 200.0     # [mm]
b   = 100.0     # [mm]
tw  = 5.6       # [mm]
tf  = 8.5       # [mm]
L   = 3000.0    # [mm]

d = h - tf      # distance between flange mid-surfaces = 191.5 mm

# Material — steel  (consistent units: mm, N, s, tonne)
E   = 210000.0    # [MPa]
nu  = 0.3
rho = 7.85e-9     # [tonne/mm³]
G   = E / (2*(1 + nu))   # shear modulus

# Section properties (approximate, for analytical comparison)
A_cs  = 2*b*tf + (h - 2*tf)*tw
Iy_cs = (b*h**3 - (b - tw)*(h - 2*tf)**3) / 12   # strong axis
Iz_cs = (2*tf*b**3 + (h - 2*tf)*tw**3) / 12       # weak axis
J_cs  = (2*b*tf**3 + (h - 2*tf)*tw**3) / 3        # St-Venant torsion (approx)

# Reference load
P_ref = 1000.0    # [N] = 1 kN reference point load (downward at midspan)

print(f"IPE200: h={h}, b={b}, tw={tw}, tf={tf} mm")
print(f"Span L = {L:.0f} mm, d = {d:.1f} mm")
print(f"A  = {A_cs:.1f} mm²,  Iy = {Iy_cs:.0f} mm⁴")
print(f"Iz = {Iz_cs:.0f} mm⁴,  J  = {J_cs:.0f} mm⁴")
print(f"G  = {G:.0f} MPa")

## Part 1 — Geometry & Mesh

Same mid-surface I-beam as the modal example: 5 rectangular surfaces
(2 × bottom flange halves, web, 2 × top flange halves) sharing edges
at the web–flange junctions.

```
    X = beam axis [0, L],  Y = flange width,  Z = depth

    Cross-section at x = 0:

      p5────────p4────────p6     z = d  (top flange)
                |
                |  web
                |
      p1────────p2────────p3     z = 0  (bottom flange)
     y=-b/2    y=0       y=b/2
```

Transfinite meshing → structured quad grid → ShellMITC4 elements.

In [ ]:
g = pyGmsh(model_name="IPE200_Buckling", verbose=True)
g.initialize()

# --- 12 corner points ---
# Near end (x = 0)
p1  = g.model.add_point(0, -b/2, 0,  lc=30)
p2  = g.model.add_point(0,  0,   0,  lc=30)
p3  = g.model.add_point(0,  b/2, 0,  lc=30)
p4  = g.model.add_point(0,  0,   d,  lc=30)
p5  = g.model.add_point(0, -b/2, d,  lc=30)
p6  = g.model.add_point(0,  b/2, d,  lc=30)

# Far end (x = L)
p7  = g.model.add_point(L, -b/2, 0,  lc=30)
p8  = g.model.add_point(L,  0,   0,  lc=30)
p9  = g.model.add_point(L,  b/2, 0,  lc=30)
p10 = g.model.add_point(L,  0,   d,  lc=30)
p11 = g.model.add_point(L, -b/2, d,  lc=30)
p12 = g.model.add_point(L,  b/2, d,  lc=30)

# --- Cross-section lines at x = 0 ---
c1 = g.model.add_line(p1, p2)    # bot-flange left
c2 = g.model.add_line(p2, p3)    # bot-flange right
c3 = g.model.add_line(p2, p4)    # web
c4 = g.model.add_line(p5, p4)    # top-flange left (p5→p4)
c5 = g.model.add_line(p4, p6)    # top-flange right

# --- Cross-section lines at x = L ---
c6  = g.model.add_line(p7,  p8)
c7  = g.model.add_line(p8,  p9)
c8  = g.model.add_line(p8,  p10)
c9  = g.model.add_line(p11, p10)
c10 = g.model.add_line(p10, p12)

# --- Longitudinal lines ---
e1 = g.model.add_line(p1,  p7)    # bottom-left edge
e2 = g.model.add_line(p2,  p8)    # bottom junction
e3 = g.model.add_line(p3,  p9)    # bottom-right edge
e4 = g.model.add_line(p4,  p10)   # top junction
e5 = g.model.add_line(p5,  p11)   # top-left edge
e6 = g.model.add_line(p6,  p12)   # top-right edge

# --- 5 Surfaces (planar quads with shared edges) ---
s_bf_l = g.model.add_plane_surface(
    g.model.add_curve_loop([c1, e2, -c6, -e1]),  label="BotFlange_L")
s_bf_r = g.model.add_plane_surface(
    g.model.add_curve_loop([c2, e3, -c7, -e2]),  label="BotFlange_R")
s_web  = g.model.add_plane_surface(
    g.model.add_curve_loop([c3, e4, -c8, -e2]),  label="Web")
s_tf_l = g.model.add_plane_surface(
    g.model.add_curve_loop([-c4, e5, c9, -e4]),  label="TopFlange_L")
s_tf_r = g.model.add_plane_surface(
    g.model.add_curve_loop([c5, e6, -c10, -e4]), label="TopFlange_R")

g.model.registry()

In [ ]:
# --- Physical groups ---
# Surfaces → element regions (section assignment)
pg_flanges = g.physical.add(2, [s_bf_l, s_bf_r, s_tf_l, s_tf_r], name="Flanges")
pg_web     = g.physical.add(2, [s_web], name="Web")

# Boundary curves
# Pin support at x = 0: all cross-section curves
pg_pin = g.physical.add(1, [c1, c2, c3, c4, c5], name="PinEnd")

# Roller support at x = L: all cross-section curves
pg_roller = g.physical.add(1, [c6, c7, c8, c9, c10], name="RollerEnd")

g.physical.summary()

In [ ]:
# --- Transfinite structured quad mesh ---
n_half_flange = 4    # across half-flange (50 mm → ~12.5 mm/elem)
n_web_height  = 12   # across web height  (191.5 mm → ~16 mm/elem)
n_length      = 60   # along beam length  (3000 mm → 50 mm/elem)

# Flange half-width curves
for c in [c1, c2, c4, c5, c6, c7, c9, c10]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_half_flange + 1)

# Web height curves
for c in [c3, c8]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_web_height + 1)

# Beam length curves
for c in [e1, e2, e3, e4, e5, e6]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_length + 1)

# Transfinite surfaces + recombine → quads
for s in [s_bf_l, s_bf_r, s_web, s_tf_l, s_tf_r]:
    gmsh.model.mesh.setTransfiniteSurface(s)
    gmsh.model.mesh.setRecombine(2, s)

g.mesh.set_order(1)
g.mesh.generate(2)

stats = g.mesh.get_elements(dim=2)
n_quads = sum(len(t) for t in stats['tags'])
print(f"Generated {n_quads} quad shell elements")

## Part 2 — Mesh Extraction & Node Classification

We extract the full mesh, classify elements by region (flange vs web),
and locate special nodes:
- **Pin-end nodes** (x = 0): fix u_x, u_y, u_z
- **Roller-end nodes** (x = L): fix u_y, u_z (free axial)
- **Load node**: closest to (L/2, 0, d) — top of web at midspan

In [ ]:
fem = g.mesh.get_fem_data(dim=2)

node_tags    = fem['node_tags']
node_coords  = fem['node_coords']
tag_to_idx   = fem['tag_to_idx']
connectivity = fem['connectivity']
elem_tags    = fem['elem_tags']
used_tags    = fem['used_tags']

# --- Region classification ---
flange_elem_set = set()
for surf in [s_bf_l, s_bf_r, s_tf_l, s_tf_r]:
    elems = g.mesh.get_elements(dim=2, tag=surf)
    for etags in elems['tags']:
        flange_elem_set.update(etags.astype(int).tolist())

web_elem_set = set()
for etags in g.mesh.get_elements(dim=2, tag=s_web)['tags']:
    web_elem_set.update(etags.astype(int).tolist())

# --- Boundary nodes ---
pin_nodes    = g.physical.get_nodes(1, pg_pin)['tags']
roller_nodes = g.physical.get_nodes(1, pg_roller)['tags']

# --- Load application node (midspan, top of web) ---
target = np.array([L/2, 0.0, d])
dists  = np.linalg.norm(node_coords - target, axis=1)
load_idx  = np.argmin(dists)
load_gtag = int(node_tags[load_idx])

print(f"Nodes: {len(node_tags)} total, {len(used_tags)} used")
print(f"Quads: {len(flange_elem_set)} flange + {len(web_elem_set)} web = {connectivity.shape[0]}")
print(f"Pin nodes (x=0): {len(pin_nodes)}, Roller nodes (x=L): {len(roller_nodes)}")
print(f"Load node: tag {load_gtag} at {node_coords[load_idx]} (dist={dists[load_idx]:.4f} mm)")

## Part 3 — OpenSees Shell Model

**Simply-supported boundary conditions:**
- x = 0 (pin): fix u_x, u_y, u_z — prevents all translation
- x = L (roller): fix u_y, u_z — allows axial expansion, prevents vertical/lateral sway

Both ends fix u_y at all cross-section nodes, which provides lateral
restraint and prevents rigid-body twist. This is the "fork support"
assumption, standard for LTB calculations.

**Point load:** P applied downward (−z) at the top flange, midspan.
Loading at the top flange is the most destabilizing configuration for
LTB (load above the shear center increases the overturning effect).

In [ ]:
def build_model():
    """Build the OpenSees shell model from scratch.
    
    Returns the OpenSees node tag for the load application point.
    Called once; the model persists until ops.wipe().
    """
    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)

    # --- Nodes ---
    gmsh_to_ops = {}
    new_id = 0
    for gtag, coords in zip(node_tags.astype(int), node_coords):
        if int(gtag) not in used_tags:
            continue
        new_id += 1
        gmsh_to_ops[int(gtag)] = new_id
        ops.node(new_id, float(coords[0]), float(coords[1]), float(coords[2]))

    # --- Sections ---
    sec_flange = 1
    sec_web    = 2
    ops.section('ElasticMembranePlateSection', sec_flange, E, nu, tf, rho)
    ops.section('ElasticMembranePlateSection', sec_web,    E, nu, tw, rho)

    # --- Elements ---
    for i, (etag, row) in enumerate(zip(elem_tags, connectivity)):
        eid = i + 1
        nodes = [gmsh_to_ops[int(n)] for n in row]
        sec = sec_flange if etag in flange_elem_set else sec_web
        ops.element('ShellMITC4', eid, *nodes, sec)

    # --- BCs: simply supported ---
    # Pin end (x = 0): fix ux, uy, uz
    for gtag in pin_nodes.astype(int):
        if int(gtag) in gmsh_to_ops:
            ops.fix(gmsh_to_ops[int(gtag)], 1, 1, 1, 0, 0, 0)

    # Roller end (x = L): fix uy, uz (free axial)
    for gtag in roller_nodes.astype(int):
        if int(gtag) in gmsh_to_ops:
            ops.fix(gmsh_to_ops[int(gtag)], 0, 1, 1, 0, 0, 0)

    load_ops_tag = gmsh_to_ops[load_gtag]
    print(f"Model: {new_id} nodes, {connectivity.shape[0]} ShellMITC4 elements")
    print(f"Load node: ops tag {load_ops_tag}")
    return gmsh_to_ops, load_ops_tag

gmsh_to_ops, load_ops_tag = build_model()

## Part 4 — Static Analysis Under Reference Load

Apply P_ref at midspan and solve. Extract:
- Vertical deflection (u_z) along the bottom junction line
- Midspan deflection for comparison with beam theory:
  δ_beam = PL³ / (48 E I_y)

In [ ]:
# --- Free vibration eigenvalues (no load yet) ---
#
# We grab these BEFORE applying any load. They are the baseline ω₀²
# for the VCT buckling analysis later.

eigenvalues_free = ops.eigen(10)
freq_free = [np.sqrt(ev)/(2*np.pi) for ev in eigenvalues_free]
print("Free vibration (no load):")
for i, f in enumerate(freq_free):
    print(f"  Mode {i+1}: f = {f:.3f} Hz")

# --- Apply reference load and solve static ---
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
ops.load(load_ops_tag, 0.0, 0.0, -P_ref, 0.0, 0.0, 0.0)   # P downward

ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('BandGeneral')
ops.test('NormDispIncr', 1.0e-8, 10)
ops.algorithm('Newton')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')

ok = ops.analyze(1)
print(f"\nStatic analysis: {'CONVERGED' if ok == 0 else 'FAILED'}")

# --- Extract displacements ---
nNode = len(node_tags)
disp_static = np.zeros((nNode, 3))
for gtag, ops_id in gmsh_to_ops.items():
    idx = tag_to_idx[gtag]
    for dof in range(3):
        disp_static[idx, dof] = ops.nodeDisp(ops_id, dof + 1)

# Midspan deflection
uz_mid = ops.nodeDisp(load_ops_tag, 3)
uz_beam = -P_ref * L**3 / (48 * E * Iy_cs)   # beam theory (negative = downward)
print(f"\nMidspan deflection (P = {P_ref:.0f} N):")
print(f"  Shell FEM:   u_z = {uz_mid:.6f} mm")
print(f"  Beam theory: u_z = {uz_beam:.6f} mm")
print(f"  Ratio:       {uz_mid/uz_beam:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# --- Deflection along the bottom junction line ---
#
# The bottom web-flange junction runs along y=0, z=0.
# Find all nodes on this line and plot uz vs x.

tol = 1e-3
junction_mask = (np.abs(node_coords[:, 1]) < tol) & (np.abs(node_coords[:, 2]) < tol)
junc_x  = node_coords[junction_mask, 0]
junc_uz = disp_static[junction_mask, 2]

# Sort by x for a clean plot
order = np.argsort(junc_x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Vertical deflection along span
ax = axes[0]
ax.plot(junc_x[order], junc_uz[order], 'b-', lw=1.5, label='Shell FEM')
# Beam theory: δ(x) for simply-supported with midspan point load
x_an = np.linspace(0, L, 200)
delta_an = np.where(
    x_an <= L/2,
    -P_ref * x_an * (3*L**2 - 4*x_an**2) / (48*E*Iy_cs),
    -P_ref * (L - x_an) * (3*L**2 - 4*(L - x_an)**2) / (48*E*Iy_cs)
)
ax.plot(x_an, delta_an, 'r--', lw=1.5, label='Euler-Bernoulli')
ax.set_xlabel('x [mm]')
ax.set_ylabel('u_z [mm]')
ax.set_title(f'Vertical deflection (P = {P_ref:.0f} N)')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) 3D deformed shape (magnified)
ax3d = fig.add_subplot(122, projection='3d')
conn_idx = np.array([[tag_to_idx[int(n)] for n in row] for row in connectivity])

max_d = np.max(np.abs(disp_static))
scale = 0.05 * L / max_d if max_d > 1e-15 else 1.0

deformed = node_coords.copy()
deformed[:, :3] += scale * disp_static

for row in conn_idx:
    verts = deformed[row]
    quad = np.vstack([verts, verts[0:1]])
    ax3d.plot(quad[:, 0], quad[:, 1], quad[:, 2], color='steelblue', lw=0.2)

ax3d.set_title(f'Deformed shape (×{scale:.0f})')
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax3d.view_init(elev=20, azim=-60)

plt.tight_layout()
plt.show()

## Part 5 — Linear Buckling via VCT

**Vibration Correlation Technique (VCT):**

1. We already have ω₀² (free vibration, no load).
2. Now we incrementally increase the point load P.
3. After each increment, call `ops.eigen()` to get ω²(P).
4. Plot ω² vs P — in the linear regime, this is a straight line.
5. Extrapolate to ω² = 0 → that's the critical load P_cr.

The physical reasoning: geometric stiffness from bending/compression
softens the structure, reducing natural frequencies. At the buckling
load, the effective stiffness drops to zero for the critical mode,
meaning ω² → 0 for that mode.

For a midspan point load on a simply-supported beam, the dominant
buckling mechanism is **lateral-torsional buckling (LTB)**: the
compressed top flange wants to buckle sideways while the tension
bottom flange resists, producing a coupled lateral + torsional mode.

In [ ]:
# --- VCT: incremental loading with eigenvalue tracking ---
#
# We already applied P_ref (1 kN) in the static step.
# Now apply additional increments and track eigenvalues.
#
# Load increment strategy:
#   - Total load = n_steps × P_step
#   - Keep it well below P_cr (stay in linear regime)

P_step  = 5000.0      # [N] = 5 kN per step
n_steps = 20           # → up to 100 kN total (plus the initial 1 kN)

load_levels = [P_ref]  # initial static load already applied
ev_history  = [ops.eigen(10)]  # eigenvalues at current (P_ref) state

print(f"VCT: tracking {10} modes over {n_steps} load increments of {P_step:.0f} N")
print(f"{'Step':>4s}  {'P_total [kN]':>12s}  {'f1 [Hz]':>10s}  {'f2 [Hz]':>10s}")
print("-" * 42)
print(f"{'0':>4s}  {P_ref/1000:12.1f}  {np.sqrt(ev_history[0][0])/(2*np.pi):10.3f}  {np.sqrt(ev_history[0][1])/(2*np.pi):10.3f}")

# We need a new load pattern for the incremental phase.
# Remove existing load and redefine with larger increments.
ops.remove('loadPattern', 1)

ops.timeSeries('Linear', 2)
ops.pattern('Plain', 2, 2)
ops.load(load_ops_tag, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0)  # unit load

# Re-setup analysis for incremental steps
ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('BandGeneral')
ops.test('NormDispIncr', 1.0e-6, 20)
ops.algorithm('Newton')

cumulative_load = P_ref
for step in range(n_steps):
    ops.integrator('LoadControl', P_step)
    ops.analysis('Static')
    ok = ops.analyze(1)
    
    if ok != 0:
        print(f"  Step {step+1}: FAILED at P = {cumulative_load + P_step:.0f} N")
        break
    
    cumulative_load += P_step
    ev = ops.eigen(10)
    load_levels.append(cumulative_load)
    ev_history.append(list(ev))
    
    f1 = np.sqrt(abs(ev[0]))/(2*np.pi)
    f2 = np.sqrt(abs(ev[1]))/(2*np.pi)
    print(f"{step+1:4d}  {cumulative_load/1000:12.1f}  {f1:10.3f}  {f2:10.3f}")

load_levels = np.array(load_levels)
ev_history  = np.array(ev_history)

print(f"\nCompleted {len(load_levels)} load levels up to {load_levels[-1]/1000:.1f} kN")

In [ ]:
# --- VCT plot: ω² vs P for the first few modes ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) ω² vs P for first 4 modes
ax = axes[0]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
P_cr_estimates = []

for m in range(min(4, ev_history.shape[1])):
    omega_sq = ev_history[:, m]
    omega_sq_0 = eigenvalues_free[m]
    
    ax.plot(load_levels/1000, omega_sq, 'o-', color=colors[m],
            markersize=3, lw=1.2, label=f'Mode {m+1}')
    
    # Linear fit to the last 10 points (or all if fewer)
    n_fit = min(10, len(load_levels))
    idx_fit = slice(-n_fit, None)
    P_fit = load_levels[idx_fit]
    w_fit = omega_sq[idx_fit]
    
    # Only extrapolate if ω² is decreasing (mode is being destabilized)
    if w_fit[-1] < w_fit[0]:
        coeffs = np.polyfit(P_fit, w_fit, 1)  # linear: ω² = a*P + b
        if coeffs[0] < 0:  # negative slope → ω² decreasing
            P_cr = -coeffs[1] / coeffs[0]    # x-intercept: ω² = 0
            P_cr_estimates.append((m+1, P_cr))
            
            # Draw extrapolation line
            P_ext = np.linspace(0, P_cr*1.05, 100)
            ax.plot(P_ext/1000, np.polyval(coeffs, P_ext), '--',
                    color=colors[m], lw=0.8, alpha=0.5)
            ax.axvline(P_cr/1000, color=colors[m], ls=':', lw=0.6, alpha=0.5)

ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('P [kN]')
ax.set_ylabel('ω² [rad²/s²]')
ax.set_title('VCT: ω² vs applied load')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (b) Frequency vs P
ax2 = axes[1]
for m in range(min(4, ev_history.shape[1])):
    freqs = np.sqrt(np.abs(ev_history[:, m])) / (2*np.pi)
    ax2.plot(load_levels/1000, freqs, 'o-', color=colors[m],
             markersize=3, lw=1.2, label=f'Mode {m+1}')

ax2.set_xlabel('P [kN]')
ax2.set_ylabel('f [Hz]')
ax2.set_title('Frequency vs applied load')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Print critical load estimates ---
print("\nVCT critical load estimates (P_cr where ω² → 0):")
print(f"{'Mode':>4s}  {'P_cr [kN]':>10s}  {'P_cr [N]':>10s}")
print("-" * 28)
for mode, pcr in P_cr_estimates:
    print(f"{mode:4d}  {pcr/1000:10.2f}  {pcr:10.0f}")

if P_cr_estimates:
    P_cr_min = min(pcr for _, pcr in P_cr_estimates)
    print(f"\nLowest P_cr = {P_cr_min/1000:.2f} kN (Mode {min(P_cr_estimates, key=lambda x: x[1])[0]})")

## Part 6 — Analytical LTB Comparison

For a simply-supported beam with midspan point load, the elastic
critical moment for lateral-torsional buckling is:

$$M_{cr} = C_1 \frac{\pi}{L} \sqrt{E I_z \, G J} \sqrt{1 + \frac{\pi^2 E I_w}{G J L^2}}$$

where C₁ ≈ 1.365 for a midspan point load, and I_w is the warping
constant. For an I-section: I_w ≈ I_z × (h − t_f)² / 4.

The critical load for the midspan point load case:
P_cr = 4 M_cr / L  (since M_max = P·L/4)

In [ ]:
# --- Analytical LTB critical load ---

C1  = 1.365          # moment gradient factor for midspan point load
Iw  = Iz_cs * d**2 / 4   # warping constant (approximate for I-section)

# Critical moment
M_cr = C1 * (np.pi / L) * np.sqrt(E * Iz_cs * G * J_cs) * \
       np.sqrt(1 + (np.pi**2 * E * Iw) / (G * J_cs * L**2))

# Critical load (M_max = P*L/4 for simply-supported, midspan point load)
P_cr_analytical = 4 * M_cr / L

print(f"Analytical LTB (Eurocode approach):")
print(f"  Iw  = {Iw:.0f} mm⁶")
print(f"  M_cr = {M_cr/1e6:.3f} kN·m")
print(f"  P_cr = {P_cr_analytical/1000:.2f} kN")

if P_cr_estimates:
    P_cr_fem = min(pcr for _, pcr in P_cr_estimates)
    ratio = P_cr_fem / P_cr_analytical
    print(f"\nFEM vs Analytical:")
    print(f"  P_cr (FEM, VCT) = {P_cr_fem/1000:.2f} kN")
    print(f"  P_cr (analytical) = {P_cr_analytical/1000:.2f} kN")
    print(f"  Ratio FEM/analytical = {ratio:.3f}")
    print(f"\nNote: differences arise from shell model capturing warping")
    print(f"restraint at supports (fork BCs) and load height effects.")

In [ ]:
# --- Extract the buckling mode shape ---
#
# The buckling mode is the one with the fastest-dropping ω².
# We already have the mode shapes from the last eigen() call
# (at the highest load level). The critical mode is the one
# that was closest to ω² = 0.

# Find the mode that dropped the most (relative change)
relative_drop = (np.array(eigenvalues_free) - ev_history[-1]) / np.array(eigenvalues_free)
critical_mode = np.argmax(relative_drop) + 1  # 1-based

print(f"Critical buckling mode: Mode {critical_mode}")
print(f"  ω² dropped by {relative_drop[critical_mode-1]*100:.1f}% from free vibration")

# Extract mode shape at the LAST load level (most destabilized)
# Note: eigenvectors are from the last ops.eigen() call
buckling_shape = np.zeros((nNode, 6))
for gtag, ops_id in gmsh_to_ops.items():
    idx = tag_to_idx[gtag]
    for dof in range(6):
        buckling_shape[idx, dof] = ops.nodeEigenvector(ops_id, critical_mode, dof + 1)

# Also extract first few mode shapes for visualization
mode_shapes_loaded = []
for m in range(1, min(7, 11)):
    phi = np.zeros((nNode, 6))
    for gtag, ops_id in gmsh_to_ops.items():
        idx = tag_to_idx[gtag]
        for dof in range(6):
            phi[idx, dof] = ops.nodeEigenvector(ops_id, m, dof + 1)
    mode_shapes_loaded.append(phi)

ops.wipe()
print(f"Extracted {len(mode_shapes_loaded)} mode shapes at P = {load_levels[-1]/1000:.1f} kN")

In [ ]:
# --- Visualize the critical buckling mode ---

fig = plt.figure(figsize=(16, 10))

# (a) Critical buckling mode — 3D
ax1 = fig.add_subplot(221, projection='3d')
phi = buckling_shape
max_d = np.max(np.abs(phi[:, :3]))
scale = 0.15 * L / max_d if max_d > 1e-15 else 1.0

deformed = node_coords.copy()
deformed[:, :3] += scale * phi[:, :3]

# Color by lateral displacement (uy)
uy = phi[:, 1]
uy_norm = (uy - uy.min()) / (uy.max() - uy.min() + 1e-15)

for row in conn_idx:
    verts = deformed[row]
    quad = np.vstack([verts, verts[0:1]])
    avg_color = np.mean(uy_norm[row])
    color = plt.cm.RdBu_r(avg_color)
    ax1.plot(quad[:, 0], quad[:, 1], quad[:, 2], color=color, lw=0.3)

ax1.set_title(f'Critical mode {critical_mode} (LTB shape)', fontsize=11)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.view_init(elev=25, azim=-60)

# (b) Top view showing lateral displacement
ax2 = fig.add_subplot(222, projection='3d')
for row in conn_idx:
    verts = deformed[row]
    quad = np.vstack([verts, verts[0:1]])
    avg_color = np.mean(uy_norm[row])
    ax2.plot(quad[:, 0], quad[:, 1], quad[:, 2],
             color=plt.cm.RdBu_r(avg_color), lw=0.3)

ax2.set_title('Top view (lateral sway)', fontsize=11)
ax2.view_init(elev=85, azim=-90)
ax2.set_xlabel('X'); ax2.set_ylabel('Y')

# (c-d) First two mode shapes
for plot_idx, (mode_idx, pos) in enumerate([(0, 223), (1, 224)]):
    ax = fig.add_subplot(pos, projection='3d')
    phi_m = mode_shapes_loaded[mode_idx]
    max_dm = np.max(np.abs(phi_m[:, :3]))
    sc = 0.12 * L / max_dm if max_dm > 1e-15 else 1.0
    
    def_m = node_coords.copy()
    def_m[:, :3] += sc * phi_m[:, :3]
    
    for row in conn_idx:
        verts = def_m[row]
        quad = np.vstack([verts, verts[0:1]])
        ax.plot(quad[:, 0], quad[:, 1], quad[:, 2], color='steelblue', lw=0.25)
    
    f_m = np.sqrt(abs(ev_history[-1, mode_idx])) / (2*np.pi)
    ax.set_title(f'Mode {mode_idx+1}: f = {f_m:.2f} Hz (at P={load_levels[-1]/1000:.0f} kN)',
                 fontsize=10)
    ax.view_init(elev=25, azim=-60)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')

plt.tight_layout()
plt.show()

## Part 7 — Gmsh Post-processing Views

In [ ]:
# --- Static deflection ---
node_tag_list = list(node_tags.astype(int))
g.view.add_node_vector("Static deflection (P={:.0f} N)".format(P_ref),
                       node_tag_list, disp_static)

# --- Buckling mode shapes ---
for m in range(min(len(mode_shapes_loaded), 6)):
    phi_m = mode_shapes_loaded[m]
    f_m = np.sqrt(abs(ev_history[-1, m]))/(2*np.pi)
    g.view.add_node_vector(
        f"Buckling Mode {m+1} (f={f_m:.2f} Hz)",
        node_tag_list, phi_m[:, :3])

# Lateral displacement magnitude for critical mode
uy_crit = np.abs(buckling_shape[:, 1])
g.view.add_node_scalar("LTB |u_y| (critical mode)", node_tag_list, uy_crit)

print(f"Created {g.view.count()} Gmsh views")

## Part 8 — Launch Gmsh GUI & Finalize

In [ ]:
# Launch Gmsh GUI to explore mode shapes interactively
# Tips:
#   View > Displacement Factor → scale the deformed shape
#   Tools > Options > View > Vector Display → switch between arrows and deformation
g.launch_gui()

In [ ]:
g.finalize()